In [0]:

# ============================================
# XML READER UTILITY
# ============================================

import os


def discover_xml_files(xml_volume_path):

    xml_files = []

    files = dbutils.fs.ls(xml_volume_path)

    for file in files:

        if file.path.lower().endswith(".xml"):

            file_name = file.name

            file_path = file.path

            table_name = os.path.splitext(
            file_name
            )[0]

            xml_files.append({

                "file_name": file_name,

                "file_path": file_path,

                "table_name": table_name

            })

    return xml_files

In [0]:
# ============================================
# XML PARSER UTILITY
# ============================================

import xml.etree.ElementTree as ET
import os


def parse_xml(xml_path):

    # Read XML content from Volume
    xml_content = dbutils.fs.head(
        xml_path,
        1000000
    )

    # Parse XML string
    root = ET.fromstring(
        xml_content
    )

    # Create ElementTree object
    tree = ET.ElementTree(
        root
    )

    # Extract filename
    file_name = os.path.basename(
        xml_path
    )

    # Extract root element
    root_element = root.tag

    return {

        "file_name": file_name,

        "root_element": root_element,

        "tree": tree,

        "root": root

    }

In [0]:
# xml structure explore 
def explore_xml_structure (
    root,
    parent = None,
    level=0,
    path=""
):
    structure = []

    current_path = (
        root.tag
        if path == ""
        else f"{path}.{root.tag}"
    )

    structure.append({
        "element_name": root.tag,
        "parent": parent,
        "level": level,
        "path": current_path
    })

    for child in root:
        structure.extend(
            explore_xml_structure(
                root = child,
                parent = root.tag,
                level = level + 1,
                path = current_path
            )
        )
    return structure

In [0]:
# ============================================
# XML FLATTENER UTILITY
# ============================================

def flatten_xml_element(
    element,
    flattened_record=None
):

    if flattened_record is None:

        flattened_record = {}

    children = list(element)

    # Leaf node
    if len(children) == 0:

        flattened_record[
            element.tag
        ] = element.text

        return flattened_record

    # Recursive traversal
    for child in children:

        flatten_xml_element(
            child,
            flattened_record
        )

    return flattened_record

# ============================================
# XML FLATTENER V2
# ============================================

def flatten_orders(root):

    records = []

    orders = root.findall("Order")

    for order in orders:

        record = flatten_xml_element(order)

        records.append(record)

    return records

In [0]:

# DATAFRAME CONVERTER UTILITY

def records_to_dataframe(records):

    if len(records) == 0:
        raise Exception(
            "No records found to convert"
        )
        
    df = spark.createDataFrame(records)
    return df